# Polynomial features in logistic regression

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
warnings.simplefilter("ignore")

X, y = make_moons(n_samples=200, noise=0.25, random_state=0)     # two interleaved half-moons
print(X.shape, np.bincount(y))

## Degree 2 turns two columns into five

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
print(poly.fit(X).get_feature_names_out(["x1", "x2"]))

## Cross-validated accuracy for several degrees

In [ ]:
for d in [1, 2, 3, 4, 5, 7, 10, 25]:
    model = make_pipeline(PolynomialFeatures(degree=d, include_bias=False), StandardScaler(),
                          LogisticRegression(C=1e4, max_iter=100000))
    cv = cross_val_score(model, X, y, cv=10).mean()
    train = model.fit(X, y).score(X, y)
    print(f"degree {d:2d}: {model[0].n_output_features_:3d} columns, train {train:.3f}, CV {cv:.3f}")

## Tests: the scaler, and the default penalty C=1

In [ ]:
from sklearn.exceptions import ConvergenceWarning
for d in [3, 10]:
    for scale in [True, False]:
        steps = [PolynomialFeatures(degree=d, include_bias=False)] + ([StandardScaler()] if scale else []) + [LogisticRegression(C=1e4, max_iter=1000)]
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always", ConvergenceWarning)
            m = make_pipeline(*steps).fit(X, y)
        print(f"degree {d:2d}, scaler {scale!s:5}: iterations {m[-1].n_iter_[0]:4d}, converged {not any(issubclass(x.category, ConvergenceWarning) for x in w)}, train {m.score(X, y):.3f}")
for d in [3, 4, 10, 25]:
    m = make_pipeline(PolynomialFeatures(degree=d, include_bias=False), StandardScaler(), LogisticRegression(max_iter=100000))
    print(f"degree {d:2d}, C=1: train {m.fit(X, y).score(X, y):.3f}, CV {cross_val_score(m, X, y, cv=10).mean():.3f}")